In [1]:
import os
import sys
from pathlib import Path

# Ensure the package root is on sys.path when running this notebook from within the vasp_inputs_mod folder.
# This allows `from vasp_inputs_mod ...` imports to work even if the notebook cwd is inside the package.
cwd = Path.cwd()
if cwd.name == "vasp_inputs_mod":
    sys.path.insert(0, str(cwd.parent))
elif (cwd / "vasp_inputs_mod").exists():
    sys.path.insert(0, str(cwd))
else:
    print(cwd)

import os
from pathlib import Path

# ── Step 1：检查变量是否存在 ──────────────────────────────────
vdw_path_str = os.environ.get("FLOW_VDW_KERNEL")

if vdw_path_str is None:
    print("❌ FLOW_VDW_KERNEL 未设置")
    print("   请在终端执行：export FLOW_VDW_KERNEL=/path/to/vdw_kernel.bindat")
    print("   或在 ~/.bashrc / ~/.zshrc 中添加该行后重新激活环境")
else:
    print(f"✅ FLOW_VDW_KERNEL 已设置")
    print(f"   值：{vdw_path_str}")

    # ── Step 2：检查路径是否存在 ──────────────────────────────
    vdw_path = Path(vdw_path_str)
    if not vdw_path.exists():
        print(f"❌ 路径不存在：{vdw_path}")
    else:
        print(f"✅ 路径存在")

        # ── Step 3：检查是否是文件（而非目录）────────────────
        if not vdw_path.is_file():
            print(f"❌ 该路径是目录，不是文件")
        else:
            print(f"✅ 是有效文件")

            # ── Step 4：检查文件名是否正确 ────────────────────
            if vdw_path.name != "vdw_kernel.bindat":
                print(f"⚠ 文件名为 {vdw_path.name!r}，期望 'vdw_kernel.bindat'")
            else:
                print(f"✅ 文件名正确：{vdw_path.name}")

            # ── Step 5：检查文件是否可读 ──────────────────────
            if os.access(vdw_path, os.R_OK):
                size_kb = vdw_path.stat().st_size / 1024
                print(f"✅ 文件可读，大小：{size_kb:.1f} KB")
            else:
                print(f"❌ 文件不可读（权限不足）")
                print(f"   请执行：chmod +r {vdw_path}")

✅ FLOW_VDW_KERNEL 已设置
   值：/data2/home/luodh/Git-workflow/Workflow_for_vasp_catalysis/vasp_inputs_mod/script/vdw_kernel.bindat
✅ 路径存在
✅ 是有效文件
✅ 文件名正确：vdw_kernel.bindat
✅ 文件可读，大小：153.4 KB


In [3]:
from vasp_inputs_mod.api import generate_inputs

preview = generate_inputs(
    "bulk_relax",
    structure="/data2/home/luodh/Test/VaspInput-Test/Bulk-calc/Bulk-optim/",   # dry_run 会跳过文件 I/O
    dry_run=True,
)

assert preview["calc_type"] == "bulk_relax"
assert "IBRION" in preview["incar"]
assert "ENCUT"  in preview["incar"]
# for key, val in sorted(preview["incar"].items()):
#     print(f"  {key:12s}: {val}")


[dry_run] INCAR preview:
  EDIFF  : 1e-06
  EDIFFG : -0.02
  ENCUT  : 520
  IBRION : 2
  ISIF   : 3
  LORBIT : 10
  LREAL  : Auto
  NSW    : 500
  POTIM  : 0.2

[dry_run] Submission script — PBS directives for calc_bulk_relax:
  #PBS -N calc_bulk_relax
  #PBS -l nodes=1:ppn=72:c72
  #PBS -l walltime=124:00:00
  #PBS -j oe
  #PBS -q low


## 2.1 `bulk_relax` — 体相结构弛豫

以 `ISIF=3`（晶胞形状、体积、离子位置全部自由）进行完整体相弛豫。
默认泛函为 PBE，`ENCUT=520`，`EDIFFG=-0.02`。

`LDAUTYPE=2`（Dudarev 简化方案，U_eff = U − J）将自动添加。
`dft_u` 支持三种等价格式：

```python
# 格式一 — 短键（推荐）
dft_u={"Fe": {"U": 4.0, "l": 2, "J": 0.0},
       "Co": {"U": 3.0, "l": 2}}

# 格式二 — VASP 标记名
dft_u={"Fe": {"LDAUU": 4.0, "LDAUL": 2, "LDAUJ": 0.0}}

# 格式三 — 标量简写（仅 U 值；默认 l=2，J=0）
dft_u={"Fe": 4.0, "Co": 3.0}
```

In [4]:
### 最简调用
structure_path = "/data2/home/luodh/Test/VaspInput-Test/Bulk-calc/Bulk-optim/"

out2 = generate_inputs(
    "bulk_relax",
    structure=structure_path,
    functional="BEEF",
    dft_u={"V": 4.0, "Co": 3.0},
    magmom={"V": 5.0, "Co": 3.0},
    output_dir="VCrO_bulk_relax"
)
print("输出目录：", out2)
### 使用 MAGMOM + DFT+U + special INCAR 参数

out4 = generate_inputs(
    "bulk_relax",
    structure=structure_path,
    functional="PBE",
    kpoints_density=50.0,
    dft_u={
        "V": {"U": 4.3, "l": 2, "J": 0.0}, #如果存在多个需要加U的元素，一定要全部写上，否则默认其他的不设置U值
    },
    magmom={"V": 5.0, "Cr": 0.0, "O": 0.6, "Li": 0.0}, #磁性设置
    incar={
        "EDIFFG": -0.02,
        "ENCUT":  520,
        "NPAR":   4,
        "ISMEAR": 0,
        "SIGMA":  0.05,
    },          #特殊的INCAR设置
)


输出目录： /data2/home/luodh/Git-workflow/Workflow_for_vasp_catalysis/vasp_inputs_mod/VCrO_bulk_relax


<string>:31: BadInputSetWarning: Overriding the POTCAR functional is generally not recommended  as it significantly affects the results of calculations and compatibility with other calculations done with the same input set. Note that some POTCAR symbols specified in the configuration file may not be available in the selected functional.


## 2.2 `slab_relax` — 表面 Slab 弛豫

以 `ISIF=2`（仅弛豫离子位置；晶胞形状和体积固定）弛豫表面 slab。
适用于洁净 slab 或含吸附质的表面。
> **关于吸附构型弛豫的说明：** 本项目没有独立的 `"adsorption_relax"` 计算类型。
> 吸附几何弛豫使用相同的 `"slab_relax"` 类型——区别仅在于输入结构（即已将吸附质
> 放置在 slab 上的 POSCAR）。

In [5]:
structure_path = "/data2/home/luodh/Test/VaspInput-Test/Slab-calc/slab-optim"

out = generate_inputs(
    "slab_relax",
    structure=structure_path,
    functional="RPBE",
    kpoints_density=25.0,
    dft_u={"Ti": 4.0, "Al": 3.0},
    magmom={"Ti": 2.0, "Al": 1.0, "O": 0.6},
    incar={
        "EDIFFG": -0.03,      # 典型表面收敛标准
        "LVHAR":  True,        # 写出静电势（用于功函数计算）
        "NPAR":   4,
        "DIPOL": "0.5 0.5 0.5", # 典型表面计算中设置偶极矩校正，方向垂直于表面（z 轴）
        "ALGO": "Normal",       # 表面计算中常用的电子迭代算法
    },
    output_dir="TiAlO_slab_relax"
)

<string>:35: BadInputSetWarning: Overriding the POTCAR functional is generally not recommended  as it significantly affects the results of calculations and compatibility with other calculations done with the same input set. Note that some POTCAR symbols specified in the configuration file may not be available in the selected functional.


## 2.3 `static_sp` — 单点静态计算

在固定几何构型下计算总能量（`NSW=0`，`IBRION=-1`）。
默认不保留电荷密度或波函数文件。

In [6]:
out = generate_inputs(
    "static_sp",
    structure=structure_path,
    #prev_dir=structure_path, #使用prev_dir中的参数设置,便不用再设置其他参数了。
    functional="HSE",          # HSE06 杂化泛函
    kpoints_density=50.0,
    incar={
        "ALGO": "All", # HSE 计算中常用的电子迭代算法
    },
    output_dir="TiAlO_static_sp"
)

out2 = generate_inputs(
    "static_dos",
    # structure=structure_path,
    prev_dir=structure_path, #使用prev_dir中的参数设置,便不用再设置其他参数了。
    functional="HSE",          # HSE06 杂化泛函,不能设置U值
    kpoints_density=50.0,
    incar={
        "EDIFF": 1E-6, # HSE 计算中常用的电子迭代算法
    },
    output_dir="TiAlO_static_dos"
)

/data2/home/luodh/anaconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3517: UserWarning: functional="HSE" with kpoints_density=50.0 is very expensive. Consider kpoints_density ≤ 30 for HSE calculations.
  if await self.run_code(code, result, async_=asy):
/data2/home/luodh/anaconda3/lib/python3.12/site-packages/pymatgen/io/vasp/sets.py:481: BadInputSetWarning: Hybrid functionals only support Algo = All, Damped, or Normal.
  incar=self.incar,
/data2/home/luodh/anaconda3/lib/python3.12/site-packages/pymatgen/io/vasp/sets.py:484: BadInputSetWarning: POTCAR data with symbol Ti_pv is not known by pymatgen to correspond with the selected user_potcar_functional='PBE'. This POTCAR is known to correspond with functionals ['PBE_52', 'PBE_54']. Please verify that you are using the right POTCARs!
  potcar="\n".join(self.potcar_symbols) if potcar_spec else self.potcar,
/data2/home/luodh/anaconda3/lib/python3.12/site-packages/pymatgen/io/vasp/sets.py:484: BadInputSetWarning: POTCA

## 2.4 `static_dos` — 态密度计算

带投影 DOS 输出的单点计算。默认设置 `LCHARG=True`、`LORBIT=11`、`NEDOS=2000`。

提供 `prev_dir` 时，引擎自动完成以下操作：
- **`structure` 可省略** — 自动从 `prev_dir/CONTCAR`（非空时优先）或 `prev_dir/POSCAR` 中提取结构。
- **继承 INCAR 和 KPOINTS** — 通过 `from_prev_calc_ecat` 从前序计算继承设置（`ENCUT` 等
  参数保留，弛豫标记 `IBRION`、`NSW` 自动替换为静态计算默认值）。
- **自动复制 WAVECAR/CHGCAR** — 前序目录中存在且非空时自动复制，
  并注入 `ICHARG=1`（有 CHGCAR）或 `ISTART=1`（仅 WAVECAR）；文件不存在时静默跳过。
- 用户 `incar=` 始终拥有最高优先级，可覆盖所有继承值和自动注入值。

`prev_dir` 对 `static_dos` 是可选项 — 也可以直接传入 `structure` 并手动设置所有 INCAR 参数。

In [7]:
# 省略 structure——自动从 prev_dir/CONTCAR 或 POSCAR 提取
out = generate_inputs(
    "static_dos", #也可以直接用“dos”使用
    prev_dir=structure_path,
    kpoints_density=70.0,
    dft_u={"Ti": {"U": 2.0, "l": 2}},
    magmom={"Ti": 3.0, "O": 0.0},
    incar={
    "NEDOS":  3001,         # 更细的 DOS 能量网格
    "EMIN":   -15.0,        # 能量窗口下限（eV）
    "EMAX":    15.0,        # 能量窗口上限（eV）
    "ISMEAR": -5,           # 四面体方法
    "SIGMA":   0.05,
    },
    output_dir="TiAlO_static_dos"
)

In [8]:
out = generate_inputs(
    "static_charge", #也可以直接用“dos”使用
    prev_dir=structure_path,
    kpoints_density=70.0,
    dft_u={"Ti": {"U": 2.0, "l": 2}},
    magmom={"Ti": 3.0, "O": 0.0},
    incar={
    "NEDOS":  3001,         # 更细的 DOS 能量网格
    "EMIN":   -15.0,        # 能量窗口下限（eV）
    "EMAX":    15.0,        # 能量窗口上限（eV）
    "ISMEAR": -5,           # 四面体方法
    "SIGMA":   0.05,
    },
    output_dir="TiAlO_static_charge"
)
#**输出文件：** `INCAR`、`KPOINTS`、`POSCAR`、`POTCAR`、`CHGCAR`、`AECCAR0`、`AECCAR2`

In [9]:
out = generate_inputs(
    "static_elf", #也可以直接用“dos”使用
    prev_dir=structure_path,
    kpoints_density=70.0,
    dft_u={"Ti": {"U": 2.0, "l": 2}},
    magmom={"Ti": 3.0, "O": 0.0},
    incar={
    "NEDOS":  3001,         # 更细的 DOS 能量网格
    "EMIN":   -15.0,        # 能量窗口下限（eV）
    "EMAX":    15.0,        # 能量窗口上限（eV）
    "ISMEAR": -5,           # 四面体方法
    "SIGMA":   0.05,
    },
    output_dir="TiAlO_static_elf"
)
### **输出文件：** `INCAR`、`KPOINTS`、`POSCAR`、`POTCAR`、`ELFCAR`

## 2.5 `lobster` — LOBSTER COHP 化学键分析

生成针对 LOBSTER 后处理配置的 VASP 单点输入集，以及 `lobsterin` 输入文件。
默认设置 `LWAVE=True`、`ISYM=0`、`NSW=0`。

提供 `prev_dir` 时，结构提取、INCAR/KPOINTS 继承以及 WAVECAR/CHGCAR 自动复制的
规则与 `static_dos` 完全相同。LOBSTER 后处理本身通常需要 WAVECAR，
但引擎不强制要求该文件存在，缺失时静默跳过。

In [10]:
out = generate_inputs(
    "lobster",
    functional="BEEF",
    kpoints_density=50.0,
    prev_dir=structure_path,
    dft_u={"Fe": {"U": 4.0, "l": 2}},
    magmom={"Fe": 5.0, "O": 0.0, "C": 0.0},
    incar={
        "ENCUT":  520,
        "NPAR":   4,
    },
    cohp_generator=[
        "from 1.8 to 2.3 type Fe type O orbitalwise",
        "from 1.1 to 1.5 type C  type O orbitalwise",
    ],
    lobsterin={
        "COHPstartEnergy": -20.0,
        "COHPendEnergy":    20.0,
    },
    output_dir="TiAlO_lobster"
)

/data2/home/luodh/anaconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3577: UserWarning: calc_type="lobster": NBANDS is not set in incar. Lobster typically requires an increased NBANDS for accurate COHP. Consider adding incar={"NBANDS": <N>}.
  exec(code_obj, self.user_global_ns, self.user_ns)
/data2/home/luodh/anaconda3/lib/python3.12/site-packages/pymatgen/io/vasp/sets.py:3074: BadInputSetWarning: Overriding the POTCAR functional is generally not recommended  as it significantly affects the results of calculations and compatibility with other calculations done with the same input set. Note that some POTCAR symbols specified in the configuration file may not be available in the selected functional.
  super().__post_init__()
<string>:36: UserWarning: Make sure that all parameters are okay! This is a brand new implementation.
Detected LDAU=True in written INCAR without any LDAUL/LDAUU/LDAUJ; forcing LDAU=False.
/data2/home/luodh/Git-workflow/Workflow_for_vasp_catalys

## 2.6 `dry_run` 参数——不写文件的测试模式

`dry_run=True` 返回预览字典，**不向磁盘写入任何文件**。`structure` 路径不会被访问，
可传入不存在的占位符路径。适用场景：
- 在提交作业前检查合并后的 INCAR
- 对参数组合进行自动化测试
- 验证 DFT+U 和 MAGMOM 值是否正确传递

In [11]:
preview = generate_inputs(
    "bulk_relax",
    structure="POSCAR",              # 文件无需存在
    functional="PBE",
    dft_u={"Fe": {"U": 4.0, "l": 2}},
    magmom={"Fe": 5.0, "O": 0.0},
    incar={"EDIFFG": -0.01, "NPAR": 4},
    dry_run=True,
)

print("calc_type:", preview["calc_type"])
print("functional:", preview["functional"])
print("INCAR：")
for k, v in sorted(preview["incar"].items()):
    print(f"  {k}: {v}")

[dry_run] INCAR preview:
  EDIFF    : 1e-06
  EDIFFG   : -0.01
  ENCUT    : 520
  IBRION   : 2
  ISIF     : 3
  ISPIN    : 2
  LDAU     : True
  LDAUJ    : {'Fe': 0.0}
  LDAUL    : {'Fe': 2}
  LDAUTYPE : 2
  LDAUU    : {'Fe': 4.0}
  LORBIT   : 10
  LREAL    : Auto
  MAGMOM   : {'Fe': 5.0, 'O': 0.0}
  NPAR     : 4
  NSW      : 500
  POTIM    : 0.2

[dry_run] Submission script — PBS directives for calc_bulk_relax:
  #PBS -N calc_bulk_relax
  #PBS -l nodes=1:ppn=72:c72
  #PBS -l walltime=124:00:00
  #PBS -j oe
  #PBS -q low
calc_type: bulk_relax
functional: PBE
INCAR：
  EDIFF: 1e-06
  EDIFFG: -0.01
  ENCUT: 520
  IBRION: 2
  ISIF: 3
  ISPIN: 2
  LDAU: True
  LDAUJ: {'Fe': 0.0}
  LDAUL: {'Fe': 2}
  LDAUTYPE: 2
  LDAUU: {'Fe': 4.0}
  LORBIT: 10
  LREAL: Auto
  MAGMOM: {'Fe': 5.0, 'O': 0.0}
  NPAR: 4
  NSW: 500
  POTIM: 0.2


## 2.10 `freq` / `freq_ir` — 振动频率计算

通过有限差分（`IBRION=5`）计算振动频率。支持两种调用方式：

- **提供 `prev_dir`** — 引擎从 `prev_dir/CONTCAR` 读取弛豫结构，并从前序 INCAR
  继承 ENCUT、EDIFF 等设置，选择性动力学标志原样保留。
- **不提供 `prev_dir`** — 直接通过 `structure` 指定结构；引擎使用工作流默认值
  设置所有 INCAR 参数。结构文件中已有的选择性动力学标志将被保留。

In [12]:
out = generate_inputs(
    "freq_ir",
    #structure="Pt111_CO_relax/CONTCAR",   # 覆盖结构；默认使用 prev_dir 中的 CONTCAR
    prev_dir=structure_path,           # 必须提供：INCAR/KPOINTS 继承来源
    functional="BEEF",
    kpoints_density=25.0,                 # 频率计算通常用 Gamma 或粗网格
    incar={
        "POTIM": 0.015,   # 有限差分位移步长（Å）；默认 0.015
        "NFREE": 2,       # 每个原子的位移次数；2=±δ，4=±δ/2 和 ±δ
        "NPAR":  4,
    },
    output_dir="TiAlO_freq_ir"
)

mode='inherit' is set, no fix atoms setIf this is unintended, ignore 


## 2.11 `nmr_cs` / `nmr_efg` — NMR 计算

NMR 化学位移（`nmr_cs`，设置 `LCHIMAG=True`）和电场梯度
（`nmr_efg`，设置 `LEFG=True`）。与所有泛函（包括 BEEF-vdW）兼容；
使用 BEEF 时，前置交换关联步骤由提交脚本自动处理，无需用户额外操作。

In [13]:
out = generate_inputs(
    "nmr_cs",
    structure=structure_path,   # 或省略并从 prev_dir 提取
    functional="BEEF",
    kpoints_density=100.0,       # NMR 需要密集 K 网格；建议 100 以上
    prev_dir=structure_path,    # 可选：继承 INCAR/KPOINTS
    incar={
        "ENCUT":   600,          # 更高截断能以提升 NMR 张量精度
        "EDIFF":   1e-8,         # NMR 计算需要非常严格的电子收敛
        "LCHIMAG": True,         # 化学位移张量（nmr_cs 默认开启）
        "NPAR":    4,
    },
    dry_run=False,
    output_dir="TiAlO_nmr_cs"
)

## 2.12 `nbo` — 自然键轨道分析

生成针对 NBO 后处理配置的单点计算输入集。与所有泛函（包括 BEEF-vdW）兼容。
提供 `prev_dir` 时从 `CONTCAR` 提取结构，并继承 INCAR/KPOINTS。

In [14]:
out = generate_inputs(
    "nbo",
    structure=structure_path,   # 显式结构；提供 prev_dir 时可省略
    prev_dir=structure_path,            # 继承 INCAR/KPOINTS；自动复制 WAVECAR/CHGCAR
    functional="BEEF",
    kpoints_density=50.0,
    incar={
        "ENCUT":  520,
        "EDIFF":  1e-6,
        "LWAVE":  True,           # 写出 WAVECAR 供 NBO 后处理使用
        "NPAR":   4,
    },
    nbo_config={
        "basis_source": "/path/to/custom_basis",   # 覆盖默认 NBO 基组路径
        "occ_1c": 1.60,           # 单中心占据阈值
        "occ_2c": 1.85,           # 双中心占据阈值
        "nbo_keywords": ["NBO", "NBOSUM", "NPA"],  # NBO 分析关键字列表
    },
)

ValueError: 配置验证失败:
  - 泛函 BEEF 不适用于计算类型 nbo

## 2.13 `neb` — NEB 过渡态搜索

> **重要提示：** `generate_inputs()` 无法接受独立的 `start_structure` 和
> `end_structure` 参数。NEB 计算请直接使用 `WorkflowEngine`。
> 该计算类型需要 VTST 补丁版 VASP。

In [16]:
from vasp_inputs_mod.workflow_engine import WorkflowEngine, WorkflowConfig, CalcType

engine = WorkflowEngine()
out = engine.run(WorkflowConfig(
    calc_type=CalcType.NEB,
    start_structure=structure_path,   # 初态结构
    end_structure=structure_path,     # 末态结构
    n_images=6,                           # 中间像数量
    use_idpp=True,                        # IDPP 插值（True）vs 线性插值（False）
    functional="PBE",
    kpoints_density=25.0,
    user_incar_overrides={
        "SPRING": -5,    # NEB 弹簧常数（eV/Å²）；负值 = 切向弹簧
        "NPAR":    4,
    },
    output_dir="neb_run/",
))

IDPPSolver failed (No module named 'pymatgen.analysis.diffusion'). Falling back to linear interpolation.
<string>:31: BadInputSetWarning: Overriding the POTCAR functional is generally not recommended  as it significantly affects the results of calculations and compatibility with other calculations done with the same input set. Note that some POTCAR symbols specified in the configuration file may not be available in the selected functional.


AttributeError: 'list' object has no attribute 'get'

## 2.15 `md_nvt` / `md_npt` — 分子动力学

NVT 正则系综（`md_nvt`，Nosé–Hoover 控温）和 NPT 等温等压系综
（`md_npt`，Langevin 热浴 + 控压，自动设置 `MDALGO=3`）。

温度、步数和时间步长通过 `incar=` 设置。正确的系综由 `calc_type` 字符串自动选择。

In [ ]:
out = generate_inputs(
    "md_nvt",
    structure=structure_path,   # 初始几何构型
    functional="PBE",
    kpoints_density=1.0,          # MD 通常使用 Gamma-only K 网格
    incar={
        "TEBEG": 300,             # 起始温度（K）——对应 VASP 标签 TEBEG
        "TEEND": 1000,            # 终止温度（K）；相等则为恒温
        "NSW":   10000,           # MD 总步数
        "POTIM": 2.0,             # 时间步长（fs）；默认 2 fs；含 H 时用 0.5 fs
        "MDALGO": 2,              # Nosé-Hoover 控温（NVT 默认）
        "SMASS": -3,              # 热浴耦合参数；-3 = 正则系综
        "NPAR":   4,
    },
)


### `md_npt` 完整调用


out = generate_inputs(
    "md_npt",
    structure="Fe_bulk/POSCAR",
    functional="PBE",
    kpoints_density=1.0,
    incar={
        "TEBEG":   300,           # 温度（K）
        "TEEND":   300,           # 与 TEBEG 相等→等温
        "NSW":     5000,          # MD 总步数
        "POTIM":   2.0,           # 时间步长（fs）；NPT 推荐 2 fs
        "PSTRESS": 0.0,           # 外部压力（kB）；0 = 零压 NPT
        "NPAR":    4,
    },
)
